# Notebook 2: Vectorization, Modeling & LLM Distillation

Notebook 2 — Vectorization, Modeling & LLM
Distillation
Filename: P3_MovieGenre-2_Modeling_<TeamName>.ipynb
Turn the plot text into features, train a logistic regression classifier for each of your
selected genres, and find out how much accuracy you lose when an LLM supplies the
training labels instead of IMDB.
Every model in this notebook is logistic regression. Holding the classifier
fixed makes the representation the only thing you are comparing, and keeps every
model explainable — on TF-IDF, each coefficient belongs to a word, which
Notebook 3 uses.
One binary classifier per selected genre. A movie can belong to several of
your genres at once, so each genre is its own yes/no problem with its own y. Every
step below that trains or scores a model does it for all 3 selected genres.
Steps 6–8 are knowledge distillation via pseudo-labeling. Pretend you never
had IMDB's genres column for the training set, and instead use an LLM as a teacher
to annotate the training plots, then train a lightweight classical model as a student
on those noisy labels. The student is cheap to run and deployable — the teacher is
expensive and slow. The question is how much accuracy you lose by substituting
an LLM annotator for real ground-truth labels.

## IMPORT PACKAGE 

In [1]:
import warnings
from pathlib import Path

import joblib
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import GridSearchCV, StratifiedKFold
from sklearn.metrics import classification_report, confusion_matrix, f1_score

sns.set_theme(style="whitegrid")
pd.set_option("display.max_columns", 50)
 
CSV_PATH = Path("mpst_imdb_movies.csv")              # same folder as the notebooks
MIN_VOTES = 100  
GENRES = ["Crime", "Action", "Comedy"]
EMB_MODEL_ID = "sentence-transformers/all-MiniLM-L6-v2"
EMB_SHORT = "emb-minilm"                   # used in filenames
EMB_CACHE_DIR = Path("emb_cache")          # cache, not committed to the repo
MODELS_DIR = Path("models")                # committed to the repo

RANDOM_STATE = 42
LR_C = 0.1
CLASS_WEIGHTS = [None, "balanced",
                 {0: 1, 1: 2}, {0: 1, 1: 3}, {0: 1, 1: 5}, {0: 1, 1: 10}]

warnings.filterwarnings("ignore", category=FutureWarning)

## LOAD DATA FILTER DATA SPLIT DATA 

In [2]:
df_all = pd.read_csv(CSV_PATH)                       # unfiltered (used for the embedding cache)
df = df_all[df_all["numVotes"] >= MIN_VOTES].copy()  # filtered

train_df = df[df["split"].isin(["train", "val"])].reset_index(drop=True)
test_df = df[df["split"] == "test"].reset_index(drop=True)

text_train = train_df["plot_synopsis"]
text_test = test_df["plot_synopsis"]

print(f"Vote filter: numVotes >= {MIN_VOTES}")
print(f"Training rows (train+val): {len(train_df)}")
print(f"Test rows:                 {len(test_df)}")


Vote filter: numVotes >= 100
Training rows (train+val): 9365
Test rows:                 2330


## GENRES 

In [3]:
rates = pd.DataFrame({
    "train_positives": train_df[GENRES].sum(),
    "train_rate_%": (100 * train_df[GENRES].mean()).round(1),
    "test_positives": test_df[GENRES].sum(),
    "test_rate_%": (100 * test_df[GENRES].mean()).round(1),
})

too_rare = rates[(rates["train_rate_%"] < 5) | (rates["test_rate_%"] < 5)]
if not too_rare.empty:
    raise ValueError(f"These genres are under 5% positives, pick others:\n{too_rare}")
rates

,train_positives,train_rate_%,test_positives,test_rate_%
Crime,2009,21.5,492,21.1
Action,2078,22.2,540,23.2
Comedy,2853,30.5,680,29.2


These three genres where picked intuitively.  I know this genres are extemely popular. For the purposes of this assignment each group is well over the 5% minimum. 

## TF IDF SCALER 

In [4]:
tfidf = TfidfVectorizer(stop_words="english", min_df=5, max_features=10000, sublinear_tf=True)
x_tfidf_train_raw = tfidf.fit_transform(text_train)      # fit on TRAIN only
x_tfidf_test_raw = tfidf.transform(text_test)            # only transform the test set

tfidf_scaler = StandardScaler(with_mean=False)           # with_mean=False keeps it sparse
x_tfidf_train = tfidf_scaler.fit_transform(x_tfidf_train_raw)
x_tfidf_test = tfidf_scaler.transform(x_tfidf_test_raw)

print("TF-IDF train shape:", x_tfidf_train.shape)
print("TF-IDF test shape: ", x_tfidf_test.shape)

TF-IDF train shape: (9365, 10000)
TF-IDF test shape:  (2330, 10000)


## EMBEDDING 

In [5]:
EMB_CACHE_DIR.mkdir(parents=True, exist_ok=True)
emb_file = EMB_CACHE_DIR / f"{EMB_SHORT}_all_movies.npz"

if emb_file.exists():
    cached = np.load(emb_file)
    emb_ids, emb_all = cached["ids"], cached["emb"]
    print("Loaded cached embeddings:", emb_all.shape)
else:
    from sentence_transformers import SentenceTransformer
    st_model = SentenceTransformer(EMB_MODEL_ID)
    emb_all = st_model.encode(df_all["plot_synopsis"].tolist(),
                              batch_size=32, show_progress_bar=True)
    emb_ids = df_all["imdb_id"].astype(str).to_numpy()
    np.savez(emb_file, ids=emb_ids, emb=emb_all)
    print("Computed and saved embeddings:", emb_all.shape)

print("Embedding dimensionality:", emb_all.shape[1])

# Look up each row of train/test by imdb_id
row_of = pd.Series(np.arange(len(emb_ids)), index=emb_ids)
x_emb_train_raw = emb_all[row_of[train_df["imdb_id"].astype(str)].to_numpy()]
x_emb_test_raw = emb_all[row_of[test_df["imdb_id"].astype(str)].to_numpy()]

emb_scaler = StandardScaler()
x_emb_train = emb_scaler.fit_transform(x_emb_train_raw)  # fit on TRAIN only
x_emb_test = emb_scaler.transform(x_emb_test_raw)

reps = {
    "tfidf": (x_tfidf_train, x_tfidf_test),
    EMB_SHORT: (x_emb_train, x_emb_test),
}

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

C:\Users\grego\anaconda3\envs\msds565\Lib\site-packages\huggingface_hub\file_download.py:137: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\grego\.cache\huggingface\hub\models--sentence-transformers--all-MiniLM-L6-v2. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)


config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/612 [00:00<?, ?B/s]

... (5 outputs left out)

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Batches:   0%|          | 0/370 [00:00<?, ?it/s]

Computed and saved embeddings: (11824, 384)
Embedding dimensionality: 384


## TUNING

In [6]:
def tune_logreg(x_tr, y_tr):
    """Pick the best class_weight with 5-fold stratified CV (macro-F1)."""
    lr = LogisticRegression(C=LR_C, solver="liblinear", max_iter=1000,
                            random_state=RANDOM_STATE)
    cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)
    grid = GridSearchCV(lr, {"class_weight": CLASS_WEIGHTS},
                        scoring="f1_macro", cv=cv, n_jobs=-1, refit=True)
    grid.fit(x_tr, y_tr)

    res = grid.cv_results_
    cv_table = pd.DataFrame({
        "class_weight": [str(p["class_weight"]) for p in res["params"]],
        "macro-F1 (mean ± std)": [f"{m:.4f} ± {s:.4f}"
                                  for m, s in zip(res["mean_test_score"], res["std_test_score"])],
        "best": ["<-- best" if r == 1 else "" for r in res["rank_test_score"]],
    })
    return grid.best_estimator_, cv_table

models = {}
for rep, (x_tr, x_te) in reps.items():
    for genre in GENRES:
        y_train = train_df[genre].to_numpy()
        model, cv_table = tune_logreg(x_tr, y_train)
        models[(rep, genre)] = model
        print(f"\n=== {rep} | {genre} ===")
        print(cv_table.to_string(index=False))
    


=== tfidf | Crime ===
 class_weight macro-F1 (mean ± std)     best
         None       0.7487 ± 0.0055         
     balanced       0.7543 ± 0.0092 <-- best
 {0: 1, 1: 2}       0.7523 ± 0.0097         
 {0: 1, 1: 3}       0.7518 ± 0.0089         
 {0: 1, 1: 5}       0.7533 ± 0.0095         
{0: 1, 1: 10}       0.7532 ± 0.0101         

=== tfidf | Action ===
 class_weight macro-F1 (mean ± std)     best
         None       0.7636 ± 0.0063         
     balanced       0.7691 ± 0.0086 <-- best
 {0: 1, 1: 2}       0.7665 ± 0.0054         
 {0: 1, 1: 3}       0.7688 ± 0.0075         
 {0: 1, 1: 5}       0.7682 ± 0.0071         
{0: 1, 1: 10}       0.7685 ± 0.0068         

=== tfidf | Comedy ===
 class_weight macro-F1 (mean ± std)     best
         None       0.7394 ± 0.0093         
     balanced       0.7405 ± 0.0116 <-- best
 {0: 1, 1: 2}       0.7400 ± 0.0108         
 {0: 1, 1: 3}       0.7372 ± 0.0077         
 {0: 1, 1: 5}       0.7351 ± 0.0065         
{0: 1, 1: 10}       0.7327 ± 

## EXPORT MODELS

Notebook 3 explains the TF-IDF models, so it needs the exact fitted objects from this notebook, not refits. We save every model trained on the IMDB labels to `models/` as `P3_N2_<rep>_logreg_<genre>.pkl`, along with the fitted `TfidfVectorizer` and its `StandardScaler`. Without those two, Notebook 3 could not rebuild the features the TF-IDF models were trained on.

In [7]:
MODELS_DIR.mkdir(parents=True, exist_ok=True)

for (rep, genre), model in models.items():
    joblib.dump(model, MODELS_DIR / f"P3_N2_{rep}_logreg_{genre.lower()}.pkl")
joblib.dump(tfidf, MODELS_DIR / "P3_N2_tfidf_vectorizer.pkl")
joblib.dump(tfidf_scaler, MODELS_DIR / "P3_N2_tfidf_scaler.pkl")

sorted(p.name for p in MODELS_DIR.glob("P3_N2_*.pkl"))

['P3_N2_emb-minilm_logreg_action.pkl',
 'P3_N2_emb-minilm_logreg_comedy.pkl',
 'P3_N2_emb-minilm_logreg_crime.pkl',
 'P3_N2_tfidf_logreg_action.pkl',
 'P3_N2_tfidf_logreg_comedy.pkl',
 'P3_N2_tfidf_logreg_crime.pkl',
 'P3_N2_tfidf_scaler.pkl',
 'P3_N2_tfidf_vectorizer.pkl']